# UCS3564 – NLP | Assignment 6: Sequence Labeling (NER) using Transformers

**Dataset:** CoNLL-2003 (`eriktks/conll2003`, BIO tags PER / ORG / LOC / MISC).
**Model:** `bert-base-cased` fine-tuned for **token classification** on CoNLL-2003 with subword label alignment.
**Evaluation:** entity-level precision / recall / F1 with **seqeval** (overall and per entity type).
**Bonus:** second task (text chunking, same backbone), confidence heatmap, NetworkX knowledge graph.

> Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
!pip install -q transformers datasets seqeval networkx matplotlib accelerate pandas

In [ ]:
import re, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import networkx as nx
import textwrap
from IPython.display import display
from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification,
                          Trainer, TrainingArguments, pipeline, set_seed)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE = "bert-base-cased"      # cased model: capitalisation is a strong NER cue
EPOCHS = 3
print("Device:", DEVICE)

## 1. Load CoNLL-2003

In [ ]:
dataset = None
for dataset_name in ["eriktks/conll2003", "lhoestq/conll2003", "tomaarsen/conll2003"]:
    try:
        dataset = load_dataset(dataset_name)
        print("Loaded:", dataset_name)
        break
    except Exception as error:
        print(f"Could not load {dataset_name}: {str(error)[:120]}")
assert dataset is not None, "CoNLL-2003 could not be loaded"

print(dataset)
ner_labels = dataset["train"].features["ner_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names
print("NER labels  :", ner_labels)
print("Chunk labels:", chunk_labels)

example = dataset["train"][0]
print("\nExample tokens  :", example["tokens"])
print("Example ner tags:", [ner_labels[t] for t in example["ner_tags"]])

In [ ]:
# Entity-type distribution in the training split
counts = {}
for tags in dataset["train"]["ner_tags"]:
    for tag in tags:
        name = ner_labels[tag]
        if name.startswith("B-"):
            counts[name[2:]] = counts.get(name[2:], 0) + 1
pd.Series(counts).plot(kind="bar", title="Entities per type (training split)", figsize=(5, 3))
plt.show()

## 2. Subword tokenization and label alignment (BIO)
WordPiece can split one word into several sub-tokens, so word-level labels must be aligned to sub-tokens:
* `[CLS]` / `[SEP]` → `-100` (ignored by the loss)
* **first** sub-token of a word → the word's label
* remaining sub-tokens → `-100` (ignored) – the entity label is predicted once per word, which keeps multi-token entities such as *United Nations* (`B-ORG I-ORG`) valid.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BACKBONE)

def tokenize_and_align_labels(examples, tag_column, label_all_tokens=False):
    tokenized = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    aligned_labels = []
    for batch_index, word_labels in enumerate(examples[tag_column]):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)                              # special tokens
            elif word_id != previous_word_id:
                label_ids.append(word_labels[word_id])              # first sub-token of the word
            else:
                label_ids.append(word_labels[word_id] if label_all_tokens else -100)
            previous_word_id = word_id
        aligned_labels.append(label_ids)
    tokenized["labels"] = aligned_labels
    return tokenized

# Visual check of the alignment on one training sentence
sample = dataset["train"][4]
encoded = tokenize_and_align_labels({"tokens": [sample["tokens"]], "ner_tags": [sample["ner_tags"]]}, "ner_tags")
alignment = pd.DataFrame({
    "sub-token": tokenizer.convert_ids_to_tokens(encoded["input_ids"][0]),
    "word_id": encoded.word_ids(0),
    "aligned label": [ner_labels[l] if l != -100 else "-100 (ignored)" for l in encoded["labels"][0]]})
display(alignment)

## 3. Fine-tune a transformer for token classification (and evaluate with seqeval)
One reusable function is used for both NER and (bonus) chunking, so both tasks share the **same backbone**.

In [ ]:
def strip_ignored(predictions, labels, label_names):
    """Remove -100 positions and convert ids to BIO strings for seqeval."""
    true_sequences, predicted_sequences = [], []
    for prediction_row, label_row in zip(predictions, labels):
        true_sequences.append([label_names[l] for l in label_row if l != -100])
        predicted_sequences.append([label_names[p] for p, l in zip(prediction_row, label_row) if l != -100])
    return true_sequences, predicted_sequences

def train_token_classifier(tag_column, label_names, output_dir):
    tokenized = dataset.map(lambda ex: tokenize_and_align_labels(ex, tag_column), batched=True,
                            remove_columns=dataset["train"].column_names)
    model = AutoModelForTokenClassification.from_pretrained(
        BACKBONE, num_labels=len(label_names),
        id2label=dict(enumerate(label_names)), label2id={l: i for i, l in enumerate(label_names)})

    def compute_metrics(eval_prediction):
        logits, labels = eval_prediction
        true_seq, pred_seq = strip_ignored(np.argmax(logits, axis=-1), labels, label_names)
        return {"precision": precision_score(true_seq, pred_seq), "recall": recall_score(true_seq, pred_seq),
                "f1": f1_score(true_seq, pred_seq)}

    arguments = TrainingArguments(output_dir=output_dir, eval_strategy="epoch", save_strategy="no",
                                  learning_rate=5e-5, per_device_train_batch_size=16,
                                  per_device_eval_batch_size=64, num_train_epochs=EPOCHS,
                                  weight_decay=0.01, seed=SEED, fp16=torch.cuda.is_available(),
                                  report_to="none", logging_steps=200)
    trainer = Trainer(model=model, args=arguments, train_dataset=tokenized["train"],
                      eval_dataset=tokenized["validation"],
                      data_collator=DataCollatorForTokenClassification(tokenizer),
                      compute_metrics=compute_metrics)
    trainer.train()

    test_output = trainer.predict(tokenized["test"])
    true_seq, pred_seq = strip_ignored(np.argmax(test_output.predictions, axis=-1), test_output.label_ids, label_names)
    trainer.save_model(output_dir); tokenizer.save_pretrained(output_dir)
    return trainer, true_seq, pred_seq

ner_trainer, ner_true, ner_pred = train_token_classifier("ner_tags", ner_labels, "ner_model")

In [ ]:
# ------------------ Entity-level evaluation on the CoNLL-2003 test split ------------------ #
print(f"OVERALL  precision={precision_score(ner_true, ner_pred):.4f}  "
      f"recall={recall_score(ner_true, ner_pred):.4f}  F1={f1_score(ner_true, ner_pred):.4f}\n")
print(classification_report(ner_true, ner_pred, digits=4))

report = classification_report(ner_true, ner_pred, output_dict=True, digits=4)
TYPE_NAMES = {"PER": "PERSON", "ORG": "ORGANIZATION", "LOC": "LOCATION", "MISC": "MISCELLANEOUS"}
per_type = pd.DataFrame({TYPE_NAMES.get(k, k): {"precision": v["precision"], "recall": v["recall"],
                                                "f1": v["f1-score"], "support": int(v["support"])}
                         for k, v in report.items() if k in TYPE_NAMES}).T.round(4)
display(per_type)
per_type[["precision", "recall", "f1"]].plot(kind="bar", figsize=(7, 3), ylim=(0, 1), title="Per-entity-type scores (test)")
plt.show()

## 4. NER on the assignment passage

In [ ]:
INPUT_TEXT = ("Barack Obama was the 44th President of the United States and worked closely with the "
              "United Nations on international policy. He visited Google's headquarters in California last year.")

ner_model = ner_trainer.model.to(DEVICE).eval()
ner_pipeline = pipeline("token-classification", model=ner_model, tokenizer=tokenizer,
                        aggregation_strategy="simple", device=0 if torch.cuda.is_available() else -1)

def extract_entities(text):
    results = []
    for entity in ner_pipeline(text):
        group = entity["entity_group"].replace("B-", "").replace("I-", "")
        results.append({"entity": text[entity["start"]:entity["end"]], "label": TYPE_NAMES.get(group, group),
                        "score": round(float(entity["score"]), 3), "start": entity["start"], "end": entity["end"]})
    return results

entities = extract_entities(INPUT_TEXT)
display(pd.DataFrame(entities)[["entity", "label", "score"]])

expected = {"Barack Obama": "PERSON", "United States": "LOCATION", "United Nations": "ORGANIZATION",
            "Google": "ORGANIZATION", "California": "LOCATION"}
found = {e["entity"]: e["label"] for e in entities}
for name, label in expected.items():
    print(("OK   " if found.get(name) == label else "DIFF ") + f"{name:15s} expected {label:13s} got {found.get(name)}")

## 5. Token-by-token re-examination (ambiguous / multi-token cases)
For each WordPiece the table shows the predicted BIO label, its probability and the **second-best label** – a close second choice marks an ambiguous token
(e.g. *United* could begin a LOCATION or an ORGANIZATION until the next word is seen).

In [ ]:
def token_level_predictions(text):
    encoded = tokenizer(text, return_tensors="pt", return_offsets_mapping=True, truncation=True)
    offsets = encoded.pop("offset_mapping")[0].tolist()
    with torch.no_grad():
        logits = ner_model(**{k: v.to(DEVICE) for k, v in encoded.items()}).logits[0]
    probabilities = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"][0])
    keep = [i for i, (start, end) in enumerate(offsets) if end != start]       # drop [CLS]/[SEP]
    return [tokens[i] for i in keep], probabilities[keep]

tokens, probabilities = token_level_predictions(INPUT_TEXT)
id2label = ner_model.config.id2label
rows = []
for token, row in zip(tokens, probabilities):
    order = row.argsort()[::-1]
    rows.append({"token": token, "label": id2label[int(order[0])], "prob": round(float(row[order[0]]), 3),
                 "2nd label": id2label[int(order[1])], "2nd prob": round(float(row[order[1]]), 3)})
token_table = pd.DataFrame(rows)
display(token_table)

In [ ]:
# Extra ambiguous sentences: same word, different entity type depending on context
ambiguous_sentences = [
    "Washington signed the bill in Washington.",
    "Jordan met Jordan's national team in Amman.",
    "Apple opened a new store in Paris, and Paris Hilton visited it.",
    "The United States Army trained with the United Nations peacekeepers.",
]
for sentence in ambiguous_sentences:
    print(sentence)
    for e in extract_entities(sentence):
        print(f"    {e['entity']:25s} -> {e['label']:13s} ({e['score']})")

In [ ]:
# A deliberately naive dictionary / rule-based tagger, for comparison with the transformer
GAZETTEER = {"Obama": "PERSON", "United": "LOCATION", "States": "LOCATION", "Nations": "LOCATION",
             "Google": "ORGANIZATION", "California": "LOCATION"}

def naive_dictionary_tagger(text):
    return [(word, GAZETTEER.get(word)) for word in text.split()]          # exact match on whitespace tokens

print("Naive dictionary tags:")
for word, tag in naive_dictionary_tagger(INPUT_TEXT):
    print(f"  {word:15s} -> {tag}")
print("\nMissed by the dictionary: 'Barack' (unseen), \"Google's\" and 'California.' (punctuation/possessive break exact match), "
      "and 'United Nations' is labelled LOCATION because the word 'United' is shared with 'United States'.")

### Why a naive dictionary / rule lookup fails here (compare with the tables above)
| Case in the passage | Dictionary / rule result | Transformer behaviour |
|---|---|---|
| **United States** vs **United Nations** – both start with *United* | A word-level lookup must pick one type for *United*; it cannot use the next word, so one of them is wrong | Self-attention sees the following token (*States* → `I-LOC`, *Nations* → `I-ORG`), so the **same first word** gets `B-LOC` or `B-ORG` depending on context; the 2nd-label column above shows the ambiguity |
| **Barack Obama** (multi-token PERSON) | Only listed names match; *Barack* alone is unknown, so the entity is split or missed | `B-PER I-PER` from capitalisation and context (*was the 44th President*) – works for unseen names |
| **Google's** | The token is `Google's` (with possessive), so exact match fails | WordPiece splits `Google`, `'`, `s`; *Google* is tagged ORG from the context *headquarters* |
| **California.** | Trailing full stop breaks the exact match | Punctuation is a separate token |
| **44th**, **President**, **international policy** | Rules based on capitalisation wrongly flag *President* | These stay `O` (the model learned titles are not entities) |
| **Google's headquarters in California** (nested-looking) | A flat dictionary cannot represent an ORG related to a LOC | BIO tagging is flat, so only the inner entities (*Google* ORG, *California* LOC) are tagged – a known limitation of flat NER |
| Same word, different types (*Washington*, *Jordan* above) | One fixed type per string | Type decided from context |

## 6. Bonus tasks
### (b) Per-token label probability heatmap

In [ ]:
heat_sentence = "He visited Google's headquarters in California last year."
heat_tokens, heat_probs = token_level_predictions(heat_sentence)
label_names_ordered = [id2label[i] for i in range(heat_probs.shape[1])]

fig, ax = plt.subplots(figsize=(8, max(4, len(heat_tokens) * 0.35)))
image = ax.imshow(heat_probs, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(label_names_ordered))); ax.set_xticklabels(label_names_ordered, rotation=45)
ax.set_yticks(range(len(heat_tokens))); ax.set_yticklabels(heat_tokens)
for i in range(heat_probs.shape[0]):
    for j in range(heat_probs.shape[1]):
        if heat_probs[i, j] >= 0.05:
            ax.text(j, i, f"{heat_probs[i, j]:.2f}", ha="center", va="center", fontsize=7)
ax.set_title("Per-token label probabilities"); fig.colorbar(image, ax=ax)
plt.tight_layout(); plt.show()

### (a) Second sequence-labeling task – text chunking (same `bert-base-cased` backbone, CoNLL-2003 chunk tags)

In [ ]:
chunk_trainer, chunk_true, chunk_pred = train_token_classifier("chunk_tags", chunk_labels, "chunk_model")
print(f"CHUNKING  precision={precision_score(chunk_true, chunk_pred):.4f}  "
      f"recall={recall_score(chunk_true, chunk_pred):.4f}  F1={f1_score(chunk_true, chunk_pred):.4f}")
print(classification_report(chunk_true, chunk_pred, digits=3))

In [ ]:
# Chunk tags for the first sentence of the passage (first sub-token of every word)
chunk_model = chunk_trainer.model.to(DEVICE).eval()
words = "Barack Obama was the 44th President of the United States".split()
encoded = tokenizer(words, is_split_into_words=True, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    chunk_ids = chunk_model(**encoded).logits[0].argmax(-1).cpu().tolist()
word_ids = encoded.word_ids(0)
seen, chunk_rows = set(), []
for position, word_id in enumerate(word_ids):
    if word_id is not None and word_id not in seen:
        seen.add(word_id)
        chunk_rows.append({"word": words[word_id], "chunk tag": chunk_model.config.id2label[chunk_ids[position]]})
display(pd.DataFrame(chunk_rows))

### (c) Minimal knowledge graph (nodes = entities, edges = co-occurrence in a sentence, edge label = text between the two entities)

In [ ]:
def build_knowledge_graph(text):
    graph = nx.Graph()
    for sentence in re.split(r"(?<=[.!?])\s+", text.strip()):
        sentence_entities = extract_entities(sentence)
        for entity in sentence_entities:
            graph.add_node(entity["entity"], type=entity["label"])
        for i in range(len(sentence_entities)):
            for j in range(i + 1, len(sentence_entities)):
                first, second = sentence_entities[i], sentence_entities[j]
                between = sentence[first["end"]:second["start"]].strip(" ,.")
                if graph.has_edge(first["entity"], second["entity"]):
                    graph[first["entity"]][second["entity"]]["weight"] += 1
                else:
                    graph.add_edge(first["entity"], second["entity"], weight=1, relation=between)
    return graph

graph = build_knowledge_graph(INPUT_TEXT)
colours = {"PERSON": "#4c78a8", "ORGANIZATION": "#f58518", "LOCATION": "#54a24b", "MISCELLANEOUS": "#b279a2"}
node_colours = [colours.get(graph.nodes[n]["type"], "grey") for n in graph.nodes]
layout = nx.spring_layout(graph, seed=SEED, k=1.6)

plt.figure(figsize=(10, 7))
nx.draw_networkx(graph, layout, node_color=node_colours, node_size=2600, font_size=9)
edge_text = {(u, v): textwrap.shorten(d["relation"], width=32, placeholder="...") for u, v, d in graph.edges(data=True)}
nx.draw_networkx_edge_labels(graph, layout, edge_labels=edge_text, font_size=7)
plt.title("Entity co-occurrence knowledge graph"); plt.axis("off"); plt.show()

for u, v, d in graph.edges(data=True):
    print(f"{u}  --[{d['relation']}]-->  {v}")

### (d) What the graph captures that an entity list misses
The edge between **Barack Obama** and **United Nations** (with the text *"was the 44th President of the United States and worked closely with the"* in between) records that these two entities are
*related within the same sentence* – a person working with an organisation – whereas a plain list (Barack Obama, United States, United Nations, Google, California) only says that the names occur.
The graph also shows that **Google** and **California** belong together but are *not* connected to the Obama entities, because they appear in a different sentence.